# FraudGuard: train in Colab

Thin runner for the pipeline in `ml/src/` (see `ml/README.md`). It:

1. gets the repo code and installs the pinned `ml/requirements.txt`,
2. reads MLflow credentials from **Colab Secrets** (never typed into cells),
3. splits the data, trains quick-scan (Isolation Forest) and deep-scan (XGBoost),
4. registers both models with alias `candidate`, and evaluates the cascade on the test split.

Every step runs as a subprocess, so the pinned library versions apply without restarting the runtime.

**Before running:** add these in the Secrets panel (key icon, left sidebar) and enable notebook access:
`MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.
Put `creditcard.csv` in Google Drive at `DATA_PATH` below.

In [ ]:
# ---- Configuration ----
REPO_URL = "https://github.com/vijayasarvajith16/FraudGuard.git"
BRANCH = "main"
CODE_SOURCE = "github"  # "github" (clone) or "drive" (use a copy of the repo at DRIVE_REPO_PATH)
DRIVE_REPO_PATH = "/content/drive/MyDrive/FraudGuard"
DATA_PATH = "/content/drive/MyDrive/fraudguard/creditcard.csv"
REPORT_DIR = "/content/drive/MyDrive/fraudguard/reports"

# Regression-gate floors for evaluate.py (set from measured results, see docs/ml-results.md).
MIN_RECALL = 0.75
MIN_PR_AUC = 0.78
MIN_PRECISION = 0.85

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess

REPO_DIR = "/content/FraudGuard"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

if CODE_SOURCE == "github":
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
elif CODE_SOURCE == "drive":
    shutil.copytree(DRIVE_REPO_PATH, REPO_DIR, ignore=shutil.ignore_patterns("node_modules", ".venv", "artifacts", "data"))
else:
    raise ValueError("CODE_SOURCE must be 'github' or 'drive'")

ML_DIR = f"{REPO_DIR}/ml"
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout or "(not a git checkout)")

In [ ]:
%pip install --quiet -r /content/FraudGuard/ml/requirements.txt

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD")
missing, no_access = [], []
for key in REQUIRED_SECRETS:
    try:
        os.environ[key] = userdata.get(key)
    except userdata.SecretNotFoundError:
        missing.append(key)
    except Exception as exc:  # NotebookAccessError: the secret exists but access is toggled off
        if type(exc).__name__ != "NotebookAccessError":
            raise
        no_access.append(key)

problems = []
if missing:
    problems.append(f"Create these Colab secrets (key icon, left sidebar): {', '.join(missing)}")
if no_access:
    problems.append(f"Turn on 'Notebook access' for: {', '.join(no_access)}")
if problems:
    raise RuntimeError(" | ".join(problems))

os.environ["MLFLOW_EXPERIMENT_NAME"] = "fraudguard"
print("MLflow tracking:", os.environ["MLFLOW_TRACKING_URI"])
assert os.path.exists(DATA_PATH), f"dataset not found at {DATA_PATH}"


In [ ]:
def run(*args):
    """Run a pipeline script from ml/, streaming output; stop the notebook on failure."""
    result = subprocess.run(["python", *args], cwd=ML_DIR)
    if result.returncode != 0:
        raise SystemExit(f"{args[0]} failed with exit code {result.returncode}")


run("src/data.py", "--data", DATA_PATH)

In [ ]:
run("src/train_quick_scan.py", "--data", DATA_PATH)

In [ ]:
run("src/train_deep_scan.py", "--data", DATA_PATH)

In [ ]:
os.makedirs(REPORT_DIR, exist_ok=True)
run(
    "src/evaluate.py", "--data", DATA_PATH,
    "--quick-model", "models:/fraudguard-quick-scan@candidate",
    "--deep-model", "models:/fraudguard-deep-scan@candidate",
    "--min-recall", str(MIN_RECALL), "--min-pr-auc", str(MIN_PR_AUC), "--min-precision", str(MIN_PRECISION),
    "--log-to-mlflow", "--output", f"{REPORT_DIR}/evaluation_report.json",
)
print(f"Report saved to {REPORT_DIR}/evaluation_report.json")

## Promote to `production` (manual)

Review the runs and the evaluation report in MLflow first. Promotion is a deliberate human
step: set `PROMOTE = True` and run the cell below. (It is automated behind the evaluation gate in Phase 14.)

In [ ]:
PROMOTE = False

if PROMOTE:
    from mlflow import MlflowClient

    client = MlflowClient()
    for name in ("fraudguard-quick-scan", "fraudguard-deep-scan"):
        version = client.get_model_version_by_alias(name, "candidate").version
        client.set_registered_model_alias(name, "production", version)
        print(f"{name}: version {version} -> @production")
else:
    print("Skipped. Set PROMOTE = True after reviewing the evaluation.")